# Predicción de Precios de Viviendas - Kaggle Competition

Este notebook predice el precio de venta de viviendas usando múltiples modelos de ML.

**Métrica de evaluación:** RMSE entre log(predicción) y log(precio real)

## Importar librerías

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import make_pipeline
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## Cargar datos

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(f"Train: {train.shape}, Test: {test.shape}")

Train: (1460, 81), Test: (1459, 80)


In [3]:
# Guardar Id para submission
test_id = test['Id']

# Variable objetivo - aplicar log para distribución normal
y = np.log1p(train['SalePrice'])

# Eliminar Id y SalePrice
train.drop(['Id', 'SalePrice'], axis=1, inplace=True)
test.drop(['Id'], axis=1, inplace=True)

# Combinar para preprocesamiento uniforme
all_data = pd.concat([train, test], axis=0, ignore_index=True)
print(f"Combined data: {all_data.shape}")

Combined data: (2919, 79)


## Análisis de datos

In [ ]:
# Ver estructura
numeric_cols = all_data.select_dtypes(
    include=['int64', 'float64']).columns.tolist()
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()
print(f"Numeric columns: {len(numeric_cols)}")
print(f"Categorical columns: {len(cat_cols)}")

Numeric columns: 36
Categorical columns: 43


In [5]:
# Valores faltantes
missing = all_data.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print(f"\nColumns with missing values: {len(missing)}")
print(missing.head(10))


Columns with missing values: 34
PoolQC         2909
MiscFeature    2814
Alley          2721
Fence          2348
MasVnrType     1766
FireplaceQu    1420
LotFrontage     486
GarageQual      159
GarageYrBlt     159
GarageCond      159
dtype: int64


## Feature Engineering

In [6]:
# Rellenar valores faltantes categóricos con 'None'
for col in cat_cols:
    all_data[col] = all_data[col].fillna('None')

# Rellenar valores faltantes numéricos con 0
for col in numeric_cols:
    all_data[col] = all_data[col].fillna(0)

In [ ]:
# Crear características nuevas
# Área total
all_data['TotalSF'] = all_data['TotalBsmtSF'] + \
    all_data['1stFlrSF'] + all_data['2ndFlrSF']

# Total de baños
all_data['TotalBath'] = all_data['FullBath'] + 0.5*all_data['HalfBath'] + \
    all_data['BsmtFullBath'] + 0.5*all_data['BsmtHalfBath']

# Antigüedad
all_data['Age'] = all_data['YrSold'] - all_data['YearBuilt']

# Remodelado
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']

# Porche total
all_data['TotalPorchSF'] = all_data['OpenPorchSF'] + \
    all_data['EnclosedPorch'] + all_data['3SsnPorch'] + all_data['ScreenPorch']

## Transformar variables sesgadas

In [ ]:
# Identificar columnas numéricas sesgadas
numeric_feats = all_data.select_dtypes(include=['int64', 'float64']).columns

# Calcular skewness
skewed_feats = all_data[numeric_feats].apply(
    lambda x: skew(x.dropna())).sort_values(ascending=False)
high_skew = skewed_feats[skewed_feats > 0.75]
print(f"High skew features: {len(high_skew)}")

High skew features: 23


In [9]:
# Aplicar log1p a features muy sesgadas
from scipy.special import boxcox1p
lam = 0.15
for feat in high_skew.index:
    all_data[feat] = boxcox1p(all_data[feat], lam)

## Codificar variables categóricas

In [10]:
# Label Encoding
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()
le = LabelEncoder()
for col in cat_cols:
    all_data[col] = le.fit_transform(all_data[col].astype(str))

print(f"Final shape: {all_data.shape}")

Final shape: (2919, 84)


## Dividir datos

In [11]:
# Dividir de nuevo
X = all_data.iloc[:len(train)]
X_test = all_data.iloc[len(train):]

print(f"X: {X.shape}, X_test: {X_test.shape}")

X: (1460, 84), X_test: (1459, 84)


## Definir función de evaluación

In [ ]:
def rmse_cv(model, X, y):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    rmse = np.sqrt(-cross_val_score(model, X, y,
                   scoring='neg_mean_squared_error', cv=kf))
    return rmse

## Entrenar modelos

In [13]:
# Escalar datos
scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

### Ridge Regression

In [ ]:
# Probar diferentes alphas
for alpha in [5, 10, 15, 20]:
    ridge = Ridge(alpha=alpha)
    score = rmse_cv(ridge, X_scaled, y)
    print(
        f"Ridge alpha={alpha}: RMSE = {score.mean():.4f} (+/- {score.std():.4f})")

Ridge alpha=5: RMSE = 0.1414 (+/- 0.0258)
Ridge alpha=10: RMSE = 0.1409 (+/- 0.0255)
Ridge alpha=15: RMSE = 0.1406 (+/- 0.0252)
Ridge alpha=20: RMSE = 0.1404 (+/- 0.0249)


In [15]:
# Mejor Ridge
ridge_best = Ridge(alpha=15)
ridge_best.fit(X_scaled, y)
ridge_pred = ridge_best.predict(X_test_scaled)

### Lasso Regression

In [ ]:
for alpha in [0.0005, 0.001, 0.002]:
    lasso = Lasso(alpha=alpha, max_iter=10000)
    score = rmse_cv(lasso, X_scaled, y)
    print(
        f"Lasso alpha={alpha}: RMSE = {score.mean():.4f} (+/- {score.std():.4f})")

Lasso alpha=0.0005: RMSE = 0.1413 (+/- 0.0245)
Lasso alpha=0.001: RMSE = 0.1390 (+/- 0.0245)
Lasso alpha=0.002: RMSE = 0.1383 (+/- 0.0237)


In [17]:
# Mejor Lasso
lasso_best = Lasso(alpha=0.001, max_iter=10000)
lasso_best.fit(X_scaled, y)
lasso_pred = lasso_best.predict(X_test_scaled)

### Gradient Boosting

In [ ]:
gbr = GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4,
                                max_features='sqrt', min_samples_leaf=15, min_samples_split=10,
                                loss='huber', random_state=42)
score = rmse_cv(gbr, X, y)
print(f"GBR: RMSE = {score.mean():.4f} (+/- {score.std():.4f})")

GBR: RMSE = 0.1230 (+/- 0.0199)


In [19]:
gbr.fit(X, y)
gbr_pred = gbr.predict(X_test)

### Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=200, max_features='sqrt',
                           min_samples_leaf=5, random_state=42, n_jobs=-1)
score = rmse_cv(rf, X, y)
print(f"RF: RMSE = {score.mean():.4f} (+/- {score.std():.4f})")

RF: RMSE = 0.1460 (+/- 0.0158)


In [21]:
rf.fit(X, y)
rf_pred = rf.predict(X_test)

## Ensemble - Promedio ponderado

In [ ]:
# Ponderar predicciones (basado en rendimiento)
final_pred = (0.2 * ridge_pred + 0.2 * lasso_pred +
              0.4 * gbr_pred + 0.2 * rf_pred)

# Convertir de log a precio real
final_pred = np.expm1(final_pred)
final_pred = np.clip(final_pred, 10000, 800000)

print(f"Predictions range: {final_pred.min():.0f} - {final_pred.max():.0f}")

Predictions range: 55486 - 547521


## Crear submission

In [23]:
submission = pd.DataFrame({
    'Id': test_id,
    'SalePrice': final_pred
})

submission.to_csv('submission.csv', index=False)
print(f"\nSubmission saved! Shape: {submission.shape}")
submission.head(10)


Submission saved! Shape: (1459, 2)


,Id,SalePrice
0,1461,122790.670727
1,1462,152914.708554
2,1463,181298.484902
3,1464,192367.167316
4,1465,189595.216056
5,1466,173242.460486
6,1467,178334.174500
7,1468,167355.915317
8,1469,187955.943434
9,1470,120839.820689


In [24]:
# Verificar archivo
print(submission.to_string(index=False))

  Id     SalePrice
1461 122790.670727
1462 152914.708554
1463 181298.484902
1464 192367.167316
1465 189595.216056
1466 173242.460486
1467 178334.174500
1468 167355.915317
1469 187955.943434
1470 120839.820689
1471 203794.483323
1472  95719.777787
1473  97944.226688
1474 150445.885763
1475 115153.721037
1476 356056.150344
1477 244893.321900
1478 288236.010583
1479 292321.040082
1480 451969.983577
1481 321111.376690
1482 202837.765423
1483 179569.177447
1484 166320.333919
1485 185051.416007
1486 198472.436124
1487 322234.452392
1488 233593.000217
1489 194144.786939
1490 226911.367816
1491 196122.832208
1492  93322.899075
1493 187254.258191
1494 290521.176455
1495 284312.230238
1496 231718.475252
1497 180935.959622
1498 159662.864963
1499 159725.637385
1500 155970.689745
1501 170761.722976
1502 152572.619715
1503 284923.770439
1504 233284.716393
1505 219876.854607
1506 185300.256366
1507 243737.300260
1508 197664.562071
1509 158607.091948
1510 143903.358499
1511 146771.803448
1512 173938.